<a href="https://colab.research.google.com/github/AriesConsulting/808s-Auto-Generations-/blob/main/Aries_Hilton%E2%80%99s_%E2%80%94_4K_UHD_Upscaled_%E2%80%94_Interactive_Video_Enhancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess
import json
import sys
import os
import glob
from google.colab import files
from IPython.display import display, HTML

# --- STEP 1: INSTALL FFmpeg/FFprobe ---
print("--- 🛠️ Setting up Environment (Installing FFmpeg) ---")
try:
    # Use ! for shell command execution in Colab
    subprocess.run(['apt-get', 'update'], check=True, capture_output=True, text=True)
    subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], check=True, capture_output=True, text=True)
    print("✅ FFmpeg installation complete.")
except subprocess.CalledProcessError as e:
    print(f"❌ Error installing FFmpeg: {e.stderr}")
    sys.exit(1)


# --- STEP 2: FILE UPLOAD ---
print("\n--- ⬆️ Upload your Video File ---")
print("A file selection window will open. Choose the video you want to enhance.")

# Upload file and get the list of uploaded filenames
uploaded = files.upload()

if not uploaded:
    print("❌ No file was uploaded. Please upload a video file to proceed.")
    sys.exit(1)

# Dynamically set the input filename to the first file uploaded
INPUT_FILENAME = list(uploaded.keys())[0]

# Automatically set the output filename
base, ext = os.path.splitext(INPUT_FILENAME)
OUTPUT_FILENAME = f"{base}_ENHANCED_4K_60FPS.mp4"
# Change extension to MP4 for wide compatibility, as MOV can sometimes cause issues
if ext.lower() not in ('.mp4', '.mov'):
     OUTPUT_FILENAME = f"{base}_ENHANCED_4K_60FPS.mp4"

print(f"\n✅ File uploaded: '{INPUT_FILENAME}'")
print(f"➡️ Output will be saved as: '{OUTPUT_FILENAME}'")


# --- STEP 3: CORE FUNCTIONS ---

def get_video_info(input_file):
    """
    Retrieves video metadata (width, height) using ffprobe.
    """
    cmd = [
        'ffprobe',
        '-v', 'error',
        '-select_streams', 'v:0',
        '-show_entries', 'stream=width,height',
        '-of', 'json',
        input_file
    ]
    try:
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
        info = json.loads(result.stdout)

        if not info.get('streams'):
            raise ValueError("Could not find video streams in the file.")

        return info['streams'][0]
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"FFprobe failed. Error output:\n{e.stderr.strip()}")
    except Exception as e:
        raise Exception(f"Error reading video file metadata: {e}")

def process_video(input_file, output_file):
    """
    Constructs and runs the FFmpeg filter chain.
    """

    print(f"\n--- 🚀 Starting Enhancement Process ---")

    try:
        meta = get_video_info(input_file)
        width = int(meta['width'])
        height = int(meta['height'])
    except Exception as e:
        print(f"❌ Failed to analyze video: {e}")
        return

    # --- 1. WATERMARK CONFIGURATION (Delogo filter) ---
    # Targets a generic bottom-right area (assuming Sora/AI watermark position)
    wm_w = 200
    wm_h = 80
    wm_x = max(0, width - wm_w - 20)
    wm_y = max(0, height - wm_h - 20)

    delogo_filter = f"delogo=x={wm_x}:y={wm_y}:w={wm_w}:h={wm_h}:show=0"

    # --- 2. UPSCALING CONFIGURATION (Scale filter) ---
    # Scales to 4K Ultra HD (3840 width), preserves aspect ratio (-1) using Lanczos for quality
    scale_filter = "scale=3840:-1:flags=lanczos"

    # --- 3. INTERPOLATION (Minterpolate filter) ---
    # Creates new intermediate frames for smoother 60 FPS playback.
    interpolate_filter = "minterpolate=fps=60:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1"

    # Combine all filters
    filter_chain = f"{delogo_filter},{scale_filter},{interpolate_filter}"

    print(f"Input Resolution: {width}x{height}")
    print(f"Target Output: 4K (3840 wide) @ 60 FPS")
    print("⚠️ This process is CPU-intensive and may take several minutes per second of video.")

    cmd = [
        'ffmpeg',
        '-i', input_file,
        '-vf', filter_chain,
        '-c:v', 'libx264',   # H.264 Codec
        '-preset', 'slow',   # Slower encoding for superior quality
        '-crf', '18',        # Quality setting: 18 is visually lossless
        '-pix_fmt', 'yuv420p', # Standard pixel format for maximum compatibility
        '-c:a', 'copy',      # Copy audio stream without re-encoding
        '-y',                # Overwrite output if exists
        output_file
    ]

    try:
        subprocess.run(cmd, check=True)
        print(f"\n\n\n🎉 SUCCESS! Video enhancement is complete.")
        print(f"The file '{output_file}' is now ready for download.")

        # --- STEP 4: AUTOMATIC DOWNLOAD ---
        print("\n--- ⬇️ Starting Automatic Download ---")
        files.download(output_file)
        print(f"✅ Download of '{output_file}' initiated. Check your browser's download manager.")

    except subprocess.CalledProcessError as e:
        print(f"\n❌ FFmpeg processing failed.")
        print(f"Error details: Return Code {e.returncode}")
    except FileNotFoundError:
        print("\n❌ Error: FFmpeg command not found.")


if __name__ == "__main__":
    if os.path.exists(INPUT_FILENAME):
        if os.path.abspath(INPUT_FILENAME) == os.path.abspath(OUTPUT_FILENAME):
            print("\nError: Input and Output filenames cannot be the same.")
        else:
            process_video(INPUT_FILENAME, OUTPUT_FILENAME)

--- 🛠️ Setting up Environment (Installing FFmpeg) ---
✅ FFmpeg installation complete.

--- ⬆️ Upload your Video File ---
A file selection window will open. Choose the video you want to enhance.


Saving copy_292CBD87-6EED-40EB-989C-43BA7CDF7061.mov to copy_292CBD87-6EED-40EB-989C-43BA7CDF7061.mov

✅ File uploaded: 'copy_292CBD87-6EED-40EB-989C-43BA7CDF7061.mov'
➡️ Output will be saved as: 'copy_292CBD87-6EED-40EB-989C-43BA7CDF7061_ENHANCED_4K_60FPS.mp4'

--- 🚀 Starting Enhancement Process ---
Input Resolution: 2560x1408
Target Output: 4K (3840 wide) @ 60 FPS
⚠️ This process is CPU-intensive and may take several minutes per second of video.



🎉 SUCCESS! Video enhancement is complete.
The file 'copy_292CBD87-6EED-40EB-989C-43BA7CDF7061_ENHANCED_4K_60FPS.mp4' is now ready for download.

--- ⬇️ Starting Automatic Download ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download of 'copy_292CBD87-6EED-40EB-989C-43BA7CDF7061_ENHANCED_4K_60FPS.mp4' initiated. Check your browser's download manager.
